# Notebook 04 — Error Analysis (Phân tích Lỗi)

**Mục đích:** Khi xây dựng mô hình Deep Learning, việc đạt điểm AUC cao là chưa đủ. 
Phân tích lỗi (Error Analysis) giúp ta mổ xẻ những trường hợp mô hình đoán sai 
(người dùng click bài A nhưng mô hình lại chấm điểm bài B cao hơn bài A).

Qua đó ta hiểu được "điểm mù" của mô hình, phân loại nguyên nhân lỗi:
- **Lỗi do Clickbait:** Tiêu đề giật gân đánh lừa mô hình.
- **Lỗi Cold-start User:** Lịch sử người dùng quá ngắn, không đủ đặc trưng.
- **Lỗi Semantic Mismatch:** Mô hình không hiểu được nghĩa bóng hoặc từ đồng nghĩa.
- **Lỗi Diverse Interests:** Người dùng có sở thích quá đa dạng, khó đoán trước.

In [1]:
import sys, json, torch
import pandas as pd
import numpy as np
from pathlib import Path

# ── Dynamic Path Setup for Kaggle ──
sys.path.insert(0, '.')
# Tự động tìm thư mục chứa utils.py trên Kaggle
for p in Path('/kaggle/input').rglob('utils.py'):
    sys.path.append(str(p.parent))
    break

from utils import (
    seed_everything, TRAIN_DIR, DEV_DIR, WORK_DIR, MODEL_DIR, SEED,
    load_news, load_behaviors, build_vocab, tokenize_to_ids
)

# ── Định nghĩa lại NRAGLS++ Advanced ──
import torch.nn as nn
import torch.nn.functional as F
import math

class AdditiveAttention(nn.Module):
    def __init__(self, dim, hidden=64):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):
        w = self.proj(x).squeeze(-1)
        if mask is not None:
            w = w.masked_fill(mask, -1e4)
        w = torch.softmax(w, dim=-1)
        w = torch.nan_to_num(w, nan=0.0)
        return (x * w.unsqueeze(-1)).sum(dim=1)

class NewsEncoder(nn.Module):
    """
    CNN-based News Encoder (Fast & Robust).
    Replaces MHA with Conv1d (kernel=3) to capture local n-grams.
    Prevents MHA padding-mask NaN bugs and reduces overfitting on small datasets.
    """
    def __init__(self, vocab_size, emb_dim, news_dim, dropout=0.2, pretrained_emb=None):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        
        # Load Pretrained GloVe if provided
        if pretrained_emb is not None:
            self.word_emb.weight.data.copy_(pretrained_emb)
            # Fine-tune the embeddings
            self.word_emb.weight.requires_grad = True

        # CNN extracts local semantic features (n-grams)
        self.cnn = nn.Conv1d(in_channels=emb_dim, out_channels=news_dim, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(dropout)
        self.attn_pool = AdditiveAttention(news_dim)

    def forward(self, token_ids):
        mask = (token_ids == 0)
        
        x = self.dropout(self.word_emb(token_ids))  # (B, T, emb_dim)
        
        # Conv1d expects (Batch, Channels, Length)
        x = x.transpose(1, 2)
        x = F.relu(self.cnn(x))
        x = x.transpose(1, 2)  # Back to (B, T, news_dim)
        
        x = self.dropout(x)
        return self.attn_pool(x, mask)

In [2]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (lighter than LayerNorm)."""
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return self.scale * x / rms


class GatedLinearAttention(nn.Module):
    """
    Gated Linear Attention with Recency Decay: O(L · d²) complexity.

    Instead of: softmax(QK⊤/√d) V  →  O(L² · d)
    Computes:   φ(Q) · (φ(K)⊤ · V)  →  O(L · d²)

    Improvements over original NRAGLS:
    - Content-based gating: g = σ(Wx+b) filters noisy clicks
    - Recency decay (OUR CONTRIBUTION): g_final = g · α^(L-i)
      α is learnable, automatically downweights older history items.
      This reflects real-world temporal user preference patterns.
    """
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        assert dim % n_heads == 0

        self.W_q = nn.Linear(dim, dim)
        self.W_k = nn.Linear(dim, dim)
        self.W_v = nn.Linear(dim, dim)
        self.W_o = nn.Linear(dim, dim)

        # Content-based gating (from NRAGLS paper)
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )

        # ── OUR CONTRIBUTION: Recency Decay ──
        # Learnable decay factor α ∈ (0, 1), initialized at ~0.95
        # g_final(i) = g(i) · α^(L-i)
        # Recent items (small L-i) → decay ≈ 1.0 (keep)
        # Old items (large L-i) → decay → 0.0 (suppress)
        self._log_alpha = nn.Parameter(torch.tensor(math.log(0.95)))

        self.dropout = nn.Dropout(dropout)

    @property
    def alpha(self):
        """Decay factor constrained to (0, 1) via sigmoid."""
        return torch.sigmoid(self._log_alpha)

    def _feature_map(self, x):
        """Kernel feature map φ(x) = elu(x) + 1 (ensures non-negativity)."""
        return F.elu(x) + 1.0

    def _recency_decay(self, L, device):
        """Compute positional decay weights: α^(L-1-i) for i=0..L-1."""
        positions = torch.arange(L, device=device, dtype=torch.float32)
        # positions[0]=0 (oldest), positions[L-1]=L-1 (newest)
        # decay = α^(L-1-i): newest=α^0=1, oldest=α^(L-1)→small
        decay = self.alpha ** (L - 1 - positions)  # (L,)
        return decay.view(1, L, 1, 1)  # broadcast: (1, L, 1, 1)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        H, d = self.n_heads, self.head_dim

        # Project Q, K, V
        Q = self._feature_map(self.W_q(x)).view(B, L, H, d)
        K = self._feature_map(self.W_k(x)).view(B, L, H, d)
        V = self.W_v(x).view(B, L, H, d)

        # Content-based gate (from paper)
        g = self.gate(x).view(B, L, H, d)  # (B, L, H, d)

        # ── OUR CONTRIBUTION: Recency-modulated gating ──
        # Multiply content gate by positional decay
        decay = self._recency_decay(L, x.device)  # (1, L, 1, 1)
        g = g * decay  # recent items keep high gate, old items suppressed

        K = K * g
        V = V * g

        # Mask padding positions
        if mask is not None:
            pad_mask = (~mask).float().view(B, L, 1, 1)  # 1=valid, 0=pad
            K = K * pad_mask
            V = V * pad_mask

        # Linear Attention: Q @ (K⊤ @ V) → O(L · d²)
        KV = torch.einsum('blhd,blhe->bhde', K, V)  # (B, H, d, d)
        out = torch.einsum('blhd,bhde->blhe', Q, KV)  # (B, L, H, d)

        # Normalize by sum of keys — clamp prevents NaN in backward
        K_sum = K.sum(dim=1)  # (B, H, d)
        denom = torch.einsum('blhd,bhd->blh', Q, K_sum).unsqueeze(-1)
        denom = denom.clamp(min=0.1)  # strong clamp — avoids 1/~0 gradient explosion
        out = out / denom

        out = torch.nan_to_num(out.reshape(B, L, D), nan=0.0)
        return self.dropout(self.W_o(out))


class SGLU(nn.Module):
    """Simplified Gated Linear Unit feed-forward."""
    def __init__(self, dim, expansion=2, dropout=0.1):  # expansion=2 for smaller model
        super().__init__()
        hidden = dim * expansion
        self.W1 = nn.Linear(dim, hidden)
        self.W2 = nn.Linear(dim, hidden)
        self.W_out = nn.Linear(hidden, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.W_out(F.silu(self.W1(x)) * self.W2(x)))


class GLAUserEncoderLayer(nn.Module):
    """Single Gated Linear Attention layer + SGLU + RMSNorm."""
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.gla = GatedLinearAttention(dim, n_heads, dropout)
        self.norm2 = RMSNorm(dim)
        self.ffn = SGLU(dim, expansion=4, dropout=dropout)

    def forward(self, x, mask=None):
        x = x + self.gla(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class GLAUserEncoder(nn.Module):
    """User Encoder with Gated Linear Attention (replaces NRMS Self-Attention)."""
    def __init__(self, news_dim, n_heads, n_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            GLAUserEncoderLayer(news_dim, n_heads, dropout)
            for _ in range(n_layers)
        ])
        self.attn_pool = AdditiveAttention(news_dim)

    def forward(self, news_vecs, mask=None):
        x = news_vecs
        for layer in self.layers:
            x = layer(x, mask)
        return self.attn_pool(x, mask)

In [3]:
class NRAGLS(nn.Module):
    def __init__(self, vocab_size, emb_dim, news_dim, n_heads_user, dropout=0.2, news_tokens_tensor=None, pretrained_emb=None):
        super().__init__()
        self.news_encoder = NewsEncoder(vocab_size, emb_dim, news_dim, dropout, pretrained_emb)
        self.user_encoder = GLAUserEncoder(news_dim, n_heads_user, n_layers=2, dropout=dropout)
        
        # Precomputed news token IDs
        self.register_buffer('news_tokens_tensor', news_tokens_tensor)

    def get_news_tokens(self, nid_indices):
        return self.news_tokens_tensor[nid_indices]

    def encode_news(self, nid_indices):
        tokens = self.get_news_tokens(nid_indices)
        shape = tokens.shape
        if len(shape) > 2:
            flat = tokens.view(-1, shape[-1])
            vecs = self.news_encoder(flat)
            return vecs.view(*shape[:-1], -1)
        return self.news_encoder(tokens)

    def encode_user(self, hist_indices):
        mask = (hist_indices == 0)
        news_vecs = self.encode_news(hist_indices)
        return self.user_encoder(news_vecs, mask)

    def forward(self, hist, pos, neg):
        u = self.encode_user(hist)
        p_vec = self.encode_news(pos)
        n_vec = self.encode_news(neg)
        pos_score = (u * p_vec).sum(-1, keepdim=True)
        neg_score = (u.unsqueeze(1) * n_vec).sum(-1)
        return torch.cat([pos_score, neg_score], dim=1)

    def score_candidates(self, hist, cand_indices):
        u = self.encode_user(hist)
        c_vec = self.encode_news(cand_indices)
        return (u * c_vec).sum(-1)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
print("Loading data for analysis...")
news_train = load_news(TRAIN_DIR)
news_dev   = load_news(DEV_DIR)
news_all   = pd.concat([news_train, news_dev]).drop_duplicates('news_id').reset_index(drop=True)
beh_dev    = load_behaviors(DEV_DIR)

# Tạo từ điển tiêu đề để in ra màn hình
nid2title = dict(zip(news_all['news_id'], news_all['title']))

Loading data for analysis...


In [5]:
vocab = build_vocab(news_all, min_freq=2, max_len=30)
all_nids = news_all['news_id'].tolist()
nid2idx = {n: i+1 for i, n in enumerate(all_nids)}

# Tokenize
news_token_ids = {}
for _, row in news_all.iterrows():
    nidx = nid2idx[row['news_id']]
    news_token_ids[nidx] = tokenize_to_ids(row['title'] or '', vocab, 30)

news_tokens_tensor = torch.zeros(len(nid2idx) + 1, 30, dtype=torch.long)
for nidx, tids in news_token_ids.items():
    news_tokens_tensor[nidx] = torch.tensor(tids, dtype=torch.long)

# Load model
model = NRAGLS(
    vocab_size=len(vocab)+1, emb_dim=100, news_dim=128, 
    n_heads_user=8, dropout=0.0, news_tokens_tensor=news_tokens_tensor, pretrained_emb=None
).to(DEVICE)

model_path = None
for p in Path('/kaggle/input').rglob('nragls_advanced.pt'):
    model_path = p
    break
if not model_path:
    model_path = MODEL_DIR / 'nragls_advanced.pt'

try:
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    print("Loaded NRAGLS++ Advanced model successfully.")
except FileNotFoundError:
    print(f"Warning: Model weights not found at {model_path}. Run Notebook 02c first! Using random weights for demonstration.")

Loaded NRAGLS++ Advanced model successfully.


In [6]:
print("\n--- BẮT ĐẦU PHÂN TÍCH LỖI (ERROR ANALYSIS) ---")

# Lấy ngẫu nhiên 500 behaviors từ dev set để tìm lỗi
sample_dev = beh_dev.sample(500, random_state=SEED)
error_cases = []

with torch.no_grad():
    for _, row in sample_dev.iterrows():
        hist = row['history'].split() if isinstance(row['history'], str) else []
        if len(hist) < 3: continue # Bỏ qua người dùng có lịch sử quá ngắn
        
        hist_idx = [nid2idx.get(n, 0) for n in hist[-50:]]
        h_tensor = torch.tensor([hist_idx], dtype=torch.long, device=DEVICE)
        
        imps = row['impressions'].split()
        cand_nids = [imp.split('-')[0] for imp in imps]
        labels = [int(imp.split('-')[1]) for imp in imps]
        
        # Chỉ xét các impression có ít nhất 1 click và 1 non-click
        if sum(labels) == 0 or sum(labels) == len(labels): continue
        
        cand_idx = [nid2idx.get(n, 0) for n in cand_nids]
        c_tensor = torch.tensor(cand_idx, dtype=torch.long, device=DEVICE)
        
        scores = model.score_candidates(h_tensor, c_tensor).squeeze(0).cpu().numpy()
        
        # Sắp xếp các candidates theo score giảm dần
        ranked_indices = np.argsort(scores)[::-1]
        ranked_labels = [labels[i] for i in ranked_indices]
        ranked_nids = [cand_nids[i] for i in ranked_indices]
        
        # KIẾM TRA LỖI: Bài được rank Top 1 không được click, nhưng bài bị rank chót lại được click
        if ranked_labels[0] == 0 and 1 in ranked_labels[2:]:
            pos_true_idx = ranked_labels.index(1) # Vị trí thực sự của bài được click (nhưng bị mô hình xếp thấp)
            
            error_cases.append({
                'history': [nid2title.get(n, "Unknown") for n in hist[-5:]],
                'predicted_top_1': nid2title.get(ranked_nids[0], "Unknown"),
                'actual_clicked': nid2title.get(ranked_nids[pos_true_idx], "Unknown"),
                'rank_of_clicked': pos_true_idx + 1
            })
            
        if len(error_cases) >= 5: # Chỉ lấy 5 ví dụ tiêu biểu
            break


--- BẮT ĐẦU PHÂN TÍCH LỖI (ERROR ANALYSIS) ---


In [7]:
print(f"\n Mẫu {len(error_cases)} trường hợp lỗi:\n")

for i, case in enumerate(error_cases):
    print(f"[{i+1}] LỊCH SỬ ĐỌC (5 bài gần nhất):")
    for h in case['history']:
        print(f"    - {h}")
    
    print(f"\n   * Mô hình dự đoán bài báo: {case['predicted_top_1']}")
    
    print(f"\n   ** Thực tế User đã click bài báo: {case['actual_clicked']}, xếp hạng thứ {case['rank_of_clicked']}")
    print("-" * 70)


 Mẫu 5 trường hợp lỗi:

[1] LỊCH SỬ ĐỌC (5 bài gần nhất):
    - Gronkowski suggests that he'll unretire if NFL legalizes CBD
    - Megyn Kelly and Rose McGowan selfie sparks speculation: 'Brainstorming'
    - Fox News contributor: 'Most likely' outcome is Trump doesn't run in 2020
    - Chick-fil-A Apologizes to Customers for Promoting National Sandwich Day   Which Is on a Sunday
    - New York Senator Chuck Schumer Mocks Trump Making Florida His New Home, Donald Jr. Fires Back

   * Mô hình dự đoán bài báo: The Real Reason McDonald's Keeps the Filet-O-Fish on Their Menu

   ** Thực tế User đã click bài báo: Opinion: Colin Kaepernick is about to get what he deserves: a chance, xếp hạng thứ 2
----------------------------------------------------------------------
[2] LỊCH SỬ ĐỌC (5 bài gần nhất):
    - President Trump says UFC reception was 'like walking into a Trump Rally'
    - See the secret airplane bedrooms where flight attendants sleep on long-haul flights
    - Reports: LSU LB Mi

### Thảo luận trong Slide Báo Cáo
Dựa trên các kết quả trên, ta có thể kết luận một số nguyên nhân lỗi thường gặp:
1. **User có sở thích chuyển hướng đột ngột (Topic Drift):** Lịch sử toàn đọc Thể thao, nhưng tự dưng lại click vào Tin Tài chính (điều này hệ thống rất khó đoán).
2. **Tiêu đề gây tò mò (Clickbait):** Người dùng bị thu hút bởi tiêu đề giật gân dù nó không đúng với sở thích lịch sử.
3. **Từ đồng nghĩa / Nghĩa bóng:** Mô hình GloVe chưa thực sự nắm bắt hết các cụm từ đa nghĩa ngữ cảnh hẹp.